In [1]:
import numpy as np
from scipy.stats import ttest_1samp, shapiro, wilcoxon, t

## 📌 Question 1 : Paired t-Test (Blood Pressure)

Blood pressure (mmHg) measured before and after medication for 8 patients:  
Before: `[140, 138, 150, 145, 142, 148, 135, 152]`  
After: `[132, 130, 142, 138, 135, 140, 128, 144]`  
At α = 0.05, test whether the medication significantly reduces blood pressure. Also compute a 95% CI for the mean reduction.

#### Step 1 : Hypotheses

Let $\mu_d$ = true mean of (before - after) differences.

$$H_0: \mu_d = 0 \quad \text{(no change)}$$
$$H_1: \mu_d > 0 \quad \text{(blood pressure decreased, one-tailed)}$$

#### Step 2 : Why this test

The same 8 patients are measured twice, so the observations are paired. A paired t-test reduces between-subject noise by working directly on the within-subject differences. With the differences treated as a single sample, this collapses to a one-sample t-test on $d_i = \text{before}_i - \text{after}_i$. Population variance is unknown, so t (not Z) is correct.

#### Step 3 : Test Statistic

$$t = \frac{\bar{d}}{s_d / \sqrt{n}}, \quad df = n - 1$$

where $\bar{d}$ is the mean difference, $s_d$ is the sample standard deviation of the differences.

In [2]:
before = np.array([140, 138, 150, 145, 142, 148, 135, 152])  # before readings
after  = np.array([132, 130, 142, 138, 135, 140, 128, 144])  # after readings
alpha  = 0.05                                                  # significance level
n      = len(before)                                           # sample size

In [3]:
d     = before - after          # paired differences
d_bar = d.mean()                # mean difference
sd    = np.std(d, ddof=1)       # sample std of differences
se    = sd / np.sqrt(n)         # standard error
df    = n - 1                   # degrees of freedom
t_stat = d_bar / se             # test statistic

#### Step 4 : Critical Value Method

In [4]:
t_crit = t.ppf(1 - alpha, df)          # one-tailed critical value
print(f't_stat : {t_stat:.3f}')
print(f't_crit : {t_crit:.3f}')

t_stat : 41.671
t_crit : 1.895


**t_stat (41.67) > t_crit (1.895) → Reject H₀**

#### Step 5 : p-value Method

In [5]:
pval = t.sf(t_stat, df)                # one-tailed p-value (upper tail)
print(f'p-val : {pval:.12f}')

p-val : 0.000000000598


**p-val ≈ 0.0000000006 < 0.05 → Reject H₀**

#### Step 6 : Confidence Interval

A 95% CI for $\mu_d$ uses the two-tailed critical value:

$$\bar{d} \pm t_{\alpha/2,\, df} \cdot \frac{s_d}{\sqrt{n}}$$

In [6]:
t_ci  = t.ppf(1 - alpha/2, df)         # two-tailed critical value for CI
moe   = t_ci * se                       # margin of error
lo    = d_bar - moe                     # lower bound
hi    = d_bar + moe                     # upper bound
print(f'95% CI : ({lo:.3f}, {hi:.3f})')

95% CI : (7.192, 8.058)


#### ✅ Final Conclusion

| Metric | Value |
|---|---|
| Mean difference $\bar{d}$ | 7.625 mmHg |
| Std deviation $s_d$ | 0.518 |
| Standard error | 0.183 |
| t statistic | 41.671 |
| t critical (one-tailed) | 1.895 |
| p-value | ≈ 0.0000000006 |
| 95% CI for $\mu_d$ | (7.192, 8.058) |
| Decision | Reject H₀ |

Strong evidence that the medication reduces blood pressure. The mean reduction is about 7.6 mmHg, and the 95% CI (7.19, 8.06) sits entirely above zero, confirming the effect is real and practically meaningful.

---

## 📌 Question 2 : Paired t-Test + Effect Size (VO₂ Max)

VO₂ max differences after a 12-week fitness program for 10 athletes:  
`[+4.2, +3.8, +5.1, +2.9, +6.0, +3.3, +4.7, +5.5, +2.1, +4.4]`  
Test at α = 0.05 (one-tailed) whether the program improved VO₂ max.  
Also compute Cohen's $d_z = \bar{d} / s_d$ and interpret the effect size.

#### Step 1 : Hypotheses

Let $\mu_d$ = true mean improvement in VO₂ max.

$$H_0: \mu_d = 0 \quad \text{(no improvement)}$$
$$H_1: \mu_d > 0 \quad \text{(program improved VO₂ max, one-tailed)}$$

#### Step 2 : Why this test

The differences are already provided directly (before minus after is implicit). Each value is the within-subject change for one athlete, so this is still a paired design. With $n = 10$ and unknown population variance, a one-sample t-test on the differences is appropriate. One-tailed because the question only asks about improvement.

#### Step 3 : Test Statistic

$$t = \frac{\bar{d}}{s_d / \sqrt{n}}, \quad df = n - 1$$

In [7]:
d      = np.array([4.2, 3.8, 5.1, 2.9, 6.0, 3.3, 4.7, 5.5, 2.1, 4.4])  # VO2 max improvements
n      = len(d)                                                             # sample size
d_bar  = d.mean()                                                           # mean improvement
sd     = np.std(d, ddof=1)                                                  # sample std
se     = sd / np.sqrt(n)                                                    # standard error
df     = n - 1                                                              # degrees of freedom
t_stat = d_bar / se                                                         # test statistic

#### Step 4 : Critical Value Method

In [8]:
t_crit = t.ppf(1 - alpha, df)          # one-tailed critical value
print(f't_stat : {t_stat:.3f}')
print(f't_crit : {t_crit:.3f}')

t_stat : 11.009
t_crit : 1.833


**t_stat (11.009) > t_crit (1.833) → Reject H₀**

#### Step 5 : p-value Method

In [9]:
pval = t.sf(t_stat, df)                # one-tailed p-value (upper tail)
print(f'p-val : {pval:.12f}')

p-val : 0.000000799635


**p-val ≈ 0.0000008 < 0.05 → Reject H₀**

#### Step 6 : Confidence Interval

95% CI for $\mu_d$:

$$\bar{d} \pm t_{\alpha/2,\, df} \cdot \frac{s_d}{\sqrt{n}}$$

In [10]:
t_ci  = t.ppf(1 - alpha/2, df)         # two-tailed critical value for CI
moe   = t_ci * se                       # margin of error
lo    = d_bar - moe                     # lower bound
hi    = d_bar + moe                     # upper bound
print(f'95% CI : ({lo:.3f}, {hi:.3f})')

95% CI : (3.337, 5.063)


#### Part (b) : Cohen's $d_z$ (Effect Size)

$$d_z = \frac{\bar{d}}{s_d}$$

Benchmarks: 0.2 = small, 0.5 = medium, 0.8 = large.

In [11]:
d_z = d_bar / sd                        # Cohen's d_z for paired design
print(f"Cohen's d_z : {d_z:.3f}")

Cohen's d_z : 3.481


#### ✅ Final Conclusion

| Metric | Value |
|---|---|
| Mean improvement $\bar{d}$ | 4.200 mL/kg/min |
| Std deviation $s_d$ | 1.206 |
| Standard error | 0.382 |
| t statistic | 11.009 |
| t critical (one-tailed) | 1.833 |
| p-value | ≈ 0.0000008 |
| 95% CI for $\mu_d$ | (3.337, 5.063) |
| Cohen's $d_z$ | 3.481 |
| Decision | Reject H₀ |

The program significantly improved VO₂ max. The mean gain of 4.2 mL/kg/min is both statistically significant and practically large. Cohen's $d_z \approx 3.48$ is far above the "large" threshold of 0.8, meaning the treatment effect is enormous relative to within-subject variability.

---

## 📌 Question 3 : Paired t-Test + Normality + Wilcoxon (Crossover Trial)

Crossover trial: 15 patients receive Drug A then Drug B (with washout period).  
Pain score differences (A−B): `[2.1, -0.5, 3.4, 1.8, 2.9, -1.2, 0.8, 4.1, 1.5, 2.7, 0.3, 3.8, 1.1, 2.5, 1.9]`  
(a) Run the paired t-test.  
(b) Test normality using Shapiro-Wilk.  
(c) Run the Wilcoxon signed-rank test.  
(d) Interpret agreement or disagreement between the two results.

#### Part (a) : Paired t-Test

#### Step 1 : Hypotheses

Let $\mu_d$ = true mean pain score difference (Drug A $-$ Drug B).

$$H_0: \mu_d = 0 \quad \text{(no difference between drugs)}$$
$$H_1: \mu_d \neq 0 \quad \text{(Drug A and Drug B differ, two-tailed)}$$

#### Step 2 : Why this test

Each patient provides one difference score (A−B) from a crossover design, so observations are paired. Working on the differences directly eliminates between-subject variability and reduces this to a one-sample t-test on $d_i$. Population variance is unknown and $n = 15$, so t (not Z) is correct. Two-tailed because the question asks whether the drugs differ, with no directional assumption.

#### Step 3 : Test Statistic

$$t = \frac{\bar{d}}{s_d / \sqrt{n}}, \quad df = n - 1$$

where $\bar{d}$ is the mean difference and $s_d$ is the sample standard deviation of the differences.

In [12]:
d      = np.array([2.1, -0.5, 3.4, 1.8, 2.9, -1.2, 0.8, 4.1, 1.5, 2.7, 0.3, 3.8, 1.1, 2.5, 1.9])  # pain score differences
alpha  = 0.05                                                                                            # significance level
n      = len(d)                                                                                          # sample size
d_bar  = d.mean()                                                                                        # mean difference
sd     = np.std(d, ddof=1)                                                                               # sample std of differences
se     = sd / np.sqrt(n)                                                                                 # standard error
df     = n - 1                                                                                           # degrees of freedom
t_stat = d_bar / se                                                                                      # test statistic

#### Step 4 : Critical Value Method

In [13]:
t_crit = t.ppf(1 - alpha/2, df)        # two-tailed critical value
print(f't_stat : {t_stat:.3f}')
print(f't_crit : {t_crit:.3f}')

t_stat : 4.604
t_crit : 2.145


**t_stat (4.604) > t_crit (2.145) -> Reject H₀**

#### Step 5 : p-value Method

In [14]:
pval = 2 * t.sf(t_stat, df)            # two-tailed p-value
print(f'p-val : {pval:.12f}')

p-val : 0.000409384466


**p-val ≈ 0.00041 < 0.05 -> Reject H₀**

#### Step 6 : Confidence Interval

95% CI for $\mu_d$:

$$\bar{d} \pm t_{\alpha/2,\, df} \cdot \frac{s_d}{\sqrt{n}}$$

In [15]:
t_ci = t.ppf(1 - alpha/2, df)          # two-tailed critical value for CI
moe  = t_ci * se                         # margin of error
lo   = d_bar - moe                       # lower bound
hi   = d_bar + moe                       # upper bound
print(f'95% CI : ({lo:.3f}, {hi:.3f})')

95% CI : (0.969, 2.658)


#### Part (b) : Shapiro-Wilk Normality Test

$$H_0: \text{differences are normally distributed}$$
$$H_1: \text{differences are not normally distributed}$$

Shapiro-Wilk tests whether the sample comes from a normal population. It returns a W statistic (closer to 1 = more normal) and a p-value.

In [16]:
W_stat, p_shapiro = shapiro(d)          # Shapiro-Wilk test on differences
print(f'W-stat    : {W_stat:.4f}')
print(f'p-val     : {p_shapiro:.4f}')

W-stat    : 0.9753
p-val     : 0.9269


**W-stat is close to 1, data looks reasonably normal** 
- **p-shapiro (0.9269) > 0.05 -> Fail to reject H₀, normality holds**

#### Part (c) : Wilcoxon Signed-Rank Test

$$H_0: \text{median difference} = 0$$
$$H_1: \text{median difference} \neq 0$$

The Wilcoxon signed-rank test is a nonparametric alternative to the paired t-test. Instead of using raw differences, it ranks their absolute values and uses the sign of each difference to compute a test statistic. It makes no normality assumption, which makes it useful when the t-test assumption is in doubt.

In [17]:
W_wilcox, p_wilcox = wilcoxon(d)       # Wilcoxon signed-rank test
print(f'W-stat : {W_wilcox:.1f}')
print(f'p-val  : {p_wilcox:.12f}')

W-stat : 7.0
p-val  : 0.001159667969


**p-wilcox (0.00116) < 0.05 -> Reject H₀**

#### Part (d) : Comparing the Two Tests

Both tests reject $H_0$ at $\alpha = 0.05$, so the conclusion is the same: Drug A and Drug B produce significantly different pain scores. The Shapiro-Wilk result (p ≈ 0.93) confirmed normality, meaning the t-test assumptions were met and the parametric result is trustworthy. The Wilcoxon test independently corroborates it, which is reassuring even though it was not strictly necessary here.

#### ✅ Final Conclusion

| Metric | Value |
|---|---|
| Mean difference $\bar{d}$ | 1.813 |
| Std deviation $s_d$ | 1.525 |
| Standard error | 0.394 |
| t statistic | 4.604 |
| t critical (two-tailed) | 2.145 |
| p-value (t-test) | ≈ 0.00041 |
| 95% CI for $\mu_d$ | (0.968, 2.659) |
| Shapiro-Wilk p-value | 0.9269 (normal) |
| Wilcoxon p-value | ≈ 0.00116 |
| Decision | Reject H₀ |

Strong evidence of a difference between Drug A and Drug B. The mean pain score is about 1.81 points higher under Drug A, and the 95% CI (0.97, 2.66) excludes zero entirely. The data passes the normality check, so the t-test result is on solid ground. The Wilcoxon test agrees, providing nonparametric confirmation.

---

## 📌 Question 4 : Paired t-Test with a Non-Zero Null (AI Productivity)

A financial analytics company introduced an AI-assisted workflow tool designed to improve analyst efficiency. The same 12 analysts were evaluated on the number of high-quality reports completed per week before and after using the tool for 8 weeks.

**Before:** `[58, 62, 55, 71, 64, 60, 59, 67, 63, 57, 61, 66]`  
**After:** `[65, 70, 61, 79, 71, 66, 63, 75, 69, 64, 68, 74]`  

Management claims the tool increases productivity by **more than 4 reports per week** on average. Assume the paired differences are approximately normally distributed. Use $\alpha = 0.01$.

(a) Perform an appropriate paired t-test to determine whether there is sufficient evidence to support this claim.  
(b) Construct a 99% confidence interval for the true mean productivity increase and interpret the result in context.

The null here isn't zero — management is claiming **more than 4**, so $\mu_0 = 4$. Same 12 analysts measured twice → paired design, work on differences. One-tailed because we only care if it exceeds 4, not just differs from it. Unknown variance + small $n$ → t distribution with $df = 11$.

#### Step 1 : Hypotheses

Let $\mu_d$ = true mean increase in reports per week (after $-$ before).

$$H_0: \mu_d \leq 4 \quad \text{(increase is at most 4 — management claim not supported)}$$
$$H_1: \mu_d > 4 \quad \text{(increase exceeds 4 — management claim supported, one-tailed)}$$

#### Step 2 : Why This Test

Each analyst contributes one before reading and one after reading, making the observations paired. Taking $d_i = \text{after}_i - \text{before}_i$ isolates the treatment effect for each individual and removes variability due to differences in baseline ability across analysts. This reduces the problem to a one-sample t-test on the differences, tested against the claimed threshold $\mu_0 = 4$. Since the population standard deviation is unknown and $n = 12$, the t distribution (not Z) applies.

#### Step 3 : Test Statistic

When $H_0$ specifies a non-zero value $\mu_0$, the test statistic becomes:

$$t = \frac{\bar{d} - \mu_0}{s_d / \sqrt{n}}, \quad df = n - 1$$

where $\mu_0 = 4$ is the threshold from the null hypothesis.

In [ ]:
before = np.array([58, 62, 55, 71, 64, 60, 59, 67, 63, 57, 61, 66])  # weekly reports before
after  = np.array([65, 70, 61, 79, 71, 66, 63, 75, 69, 64, 68, 74])  # weekly reports after
alpha  = 0.01                                                           # significance level
mu0    = 4                                                              # claimed threshold
n      = len(before)                                                    # sample size

In [ ]:
d      = after - before          # within-subject differences
d_bar  = d.mean()                # mean difference
sd     = np.std(d, ddof=1)       # sample std of differences
se     = sd / np.sqrt(n)         # standard error
df     = n - 1                   # degrees of freedom
t_stat = (d_bar - mu0) / se      # test statistic (shifted by mu0)

#### Step 4 : Critical Value Method

In [ ]:
t_crit = t.ppf(1 - alpha, df)          # one-tailed critical value
print(f't_stat : {t_stat:.3f}')
print(f't_crit : {t_crit:.3f}')

**t_stat (8.224) > t_crit (2.718) → Reject H₀**

#### Step 5 : p-value Method

In [ ]:
pval = t.sf(t_stat, df)                # one-tailed p-value (right tail)
print(f'p-val : {pval:.10f}')

**p-val ≈ 0.0000025 < 0.01 → Reject H₀**

#### Step 6 : Verification with scipy

We can verify using `ttest_1samp` with `popmean=4` to confirm our manual calculation:

In [ ]:
t_stat_, p_val_ = ttest_1samp(d, popmean=mu0, alternative='greater')
print(f't_stat : {t_stat_:.3f}')
print(f'p-val  : {p_val_:.10f}')

#### Step 7 : Confidence Interval

A 99% CI for $\mu_d$ uses the two-tailed critical value at $\alpha = 0.01$:

$$\bar{d} \pm t_{\alpha/2,\, df} \cdot \frac{s_d}{\sqrt{n}}$$

Note: the CI is for the true mean increase, not a test of whether it exceeds 4. We build it around $\bar{d}$, not around $\mu_0$.

In [ ]:
t_crit_ci = t.ppf(1 - alpha/2, df)     # two-tailed critical value for CI
moe       = t_crit_ci * se              # margin of error
lo        = d_bar - moe                 # lower bound
hi        = d_bar + moe                 # upper bound
print(f'99% CI : ({lo:.3f}, {hi:.3f})')

#### ✅ Final Conclusion

| Metric | Value |
|---|---|
| Mean increase $\bar{d}$ | 6.833 reports/week |
| Std deviation $s_d$ | 1.193 |
| Standard error | 0.345 |
| Null threshold $\mu_0$ | 4 |
| t statistic | 8.224 |
| t critical (one-tailed, α = 0.01) | 2.718 |
| p-value | ≈ 0.0000025 |
| 99% CI for $\mu_d$ | (5.763, 7.903) |
| Decision | Reject H₀ |

There is strong evidence to support management's claim. The AI tool increased productivity by an average of 6.83 reports per week, well above the claimed threshold of 4. The 99% CI (5.76, 7.90) lies entirely above 4, meaning we can be 99% confident the true average increase exceeds the management benchmark. The extremely small p-value (≈ 0.0000025) leaves virtually no room for doubt.